# 01 — Raw folder to Stage 1 HDF5

This notebook is the ingestion entry point for a new dataset. It discovers raw files, joins patient labels, previews the actual parser output, builds a validated manifest, and writes one or more nested cell-count levels to `stage1_raw_data.h5`.

## What is read

- **FCS (`.fcs`)**: read with FlowIO. Events become `[cells, channels]`; marker names use FCS `PnS` and fall back to `PnN`. Install with `pip install -e '.[fcs,notebook]'`. Compensation and biological transforms are intentionally **not** guessed here; apply them explicitly during preprocessing.
- **CSV/TSV/TXT**: rows are cells and columns are markers. A text header supplies marker names. Headerless numeric files need `markers_by_tube`.
- **NPY**: a 2-D numeric `[cells, markers]` array. Supply `markers_by_tube` for meaningful names.
- **NPZ**: use key `cells` (or the first array); an optional `markers` array supplies names.

Labels are never inferred from FCS metadata. They are joined by `patient_id` from a separate clinical/label CSV, so one patient receives one consistent label across all tubes. `counts` records the original number of events before subsampling.

## Expected input layout

The folders may be nested differently; only the regular expression below must match each path relative to `raw_root`. A typical layout is:

```text
project/
├── raw/BLAST110/
│   ├── P001_T1.fcs
│   ├── P001_T2.fcs
│   └── P002_T1.csv
├── metadata/blast110_labels.csv   # patient_id,label
└── manifests/                     # generated here
```

The raw folder may contain a mixture of supported formats. Every supported file must match the naming rule when `STRICT_FILENAMES=True`.

In [ ]:
from collections import Counter
from pathlib import Path
import re

import h5py
import numpy as np
import pandas as pd

from flowlot.io import (
    audit_manifest,
    audit_stage1,
    build_stage1_from_manifest,
    create_manifest_from_folder,
    load_cytometry_file,
)
from flowlot.io.manifest import SUPPORTED_INPUTS

## 1. Edit this configuration for your dataset

This is the only required edit for a normally structured new cohort. Named regex groups `patient_id` and `tube_id` are required. The example matches `P001_T1.fcs`. If labels are encoded in folder names instead, omit `labels_csv` and add a named `(?P<label>...)` group to the regex.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').exists() and (PROJECT_ROOT.parent / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
print('Project root:', PROJECT_ROOT)

DATASETS = [
    {
        'dataset': 'BLAST110',
        'raw_root': PROJECT_ROOT / 'raw/BLAST110',
        'labels_csv': PROJECT_ROOT / 'metadata/blast110_labels.csv',
        'manifest': PROJECT_ROOT / 'manifests/blast110.csv',
        'filename_pattern': r'(?P<patient_id>[^/]+)_(?P<tube_id>T[0-9]+)\.(?:fcs|csv|tsv|txt|npy|npz)$',
        # Needed for NPY/headerless text; FCS/headered CSV usually infer names.
        'markers_by_tube': {
            # 'T1': ['FSC-A', 'SSC-A', 'CD45', 'CD34'],
        },
        'cell_counts': [500, 1000, 2000],  # or ['all']
    },
]

STAGE1 = PROJECT_ROOT / 'data/stage1_raw_data.h5'
SEED = 42
RECURSIVE = True
STRICT_FILENAMES = True
GENERATE_MANIFESTS = False  # inspect discovery first, then set True once
OVERWRITE_MANIFESTS = False
RUN_BUILD = False           # audit/preview first, then set True

The label table must have exactly these key columns (extra clinical columns are allowed):

In [ ]:
pd.DataFrame({'patient_id': ['P001', 'P002'], 'label': ['AML', 'Healthy']})

## 2. Discover files before writing anything

This dry run shows which folder is read, which files are eligible, and which names fail the patient/tube rule.

In [ ]:
for cfg in DATASETS:
    root = cfg['raw_root'].resolve()
    if not root.is_dir():
        print(f"MISSING raw_root: {root}")
        continue
    iterator = root.rglob('*') if RECURSIVE else root.glob('*')
    files = sorted(p for p in iterator if p.is_file() and p.suffix.lower() in SUPPORTED_INPUTS)
    pattern = re.compile(cfg['filename_pattern'])
    matched = [p for p in files if pattern.search(p.relative_to(root).as_posix())]
    unmatched = [p.relative_to(root).as_posix() for p in files if p not in matched]
    print(f"\n{cfg['dataset']}: reading {root}")
    print(f"supported={len(files)}, matched={len(matched)}, formats={dict(Counter(p.suffix.lower() for p in files))}")
    print('matched examples:', [p.relative_to(root).as_posix() for p in matched[:8]])
    if unmatched:
        print('UNMATCHED examples:', unmatched[:8])

## 3. Generate the manifest

Set `GENERATE_MANIFESTS=True` only after discovery looks correct. Paths are stored relative to the manifest, making the project movable. Generation refuses to overwrite an existing manifest unless explicitly enabled.

In [ ]:
if GENERATE_MANIFESTS:
    for cfg in DATASETS:
        frame = create_manifest_from_folder(
            raw_root=cfg['raw_root'],
            output=cfg['manifest'],
            filename_pattern=cfg['filename_pattern'],
            labels=cfg.get('labels_csv'),
            markers_by_tube=cfg.get('markers_by_tube'),
            recursive=RECURSIVE,
            strict=STRICT_FILENAMES,
            overwrite=OVERWRITE_MANIFESTS,
        )
        print(f"wrote {cfg['manifest']} ({len(frame)} patient/tube rows)")
else:
    print('Dry run only: set GENERATE_MANIFESTS=True after reviewing discovery.')

## 4. Audit labels, tubes, paths, and marker declarations

The audit catches missing files, duplicate patient/tube rows, conflicting patient labels, non-finite values in the manifest, and invalid marker declarations.

In [ ]:
for cfg in DATASETS:
    if not cfg['manifest'].exists():
        print(f"MISSING manifest: {cfg['manifest']}")
        continue
    report = audit_manifest(cfg['manifest'])
    print(f"\n{cfg['dataset']} audit:", report.summary())
    display(pd.read_csv(cfg['manifest']).head())

## 5. Parse a few real files before building HDF5

This invokes the same reader as the builder and verifies matrix shape, inferred marker names, finite-value fraction, and original event count. Increase `PREVIEW_FILES` cautiously for very large FCS files.

In [ ]:
PREVIEW_FILES = 3
preview_rows = []
for cfg in DATASETS:
    manifest_path = cfg['manifest'].resolve()
    if not manifest_path.exists():
        continue
    manifest = pd.read_csv(manifest_path, dtype=str).fillna('')
    for row in manifest.head(PREVIEW_FILES).itertuples(index=False):
        source = Path(row.path)
        if not source.is_absolute():
            source = manifest_path.parent / source
        declared = [m.strip() for m in row.markers.split(';') if m.strip()] or None
        try:
            cells, markers = load_cytometry_file(source, declared)
            preview_rows.append({
                'dataset': cfg['dataset'], 'patient_id': row.patient_id, 'tube_id': row.tube_id,
                'file': source.name, 'shape': cells.shape, 'original_count': len(cells),
                'finite_fraction': float(np.isfinite(cells).mean()),
                'markers': markers[:8], 'status': 'OK',
            })
        except Exception as exc:
            preview_rows.append({'dataset': cfg['dataset'], 'file': source.name, 'status': repr(exc)})
display(pd.DataFrame(preview_rows))

## 6. Expand and build every requested cell count

`cell_counts: [500, 1000, 2000]` creates three HDF5 levels. With the same seed, smaller levels are nested subsets of larger levels (`500 ⊂ 1000 ⊂ 2000`) whenever enough cells exist. The first write creates the file and later writes append new levels. Existing target groups are protected unless `overwrite=True` is passed deliberately.

In [ ]:
CONFIGURATIONS = [
    {'dataset': cfg['dataset'], 'manifest': cfg['manifest'], 'cells': cells}
    for cfg in DATASETS
    for cells in cfg['cell_counts']
]
pd.DataFrame(CONFIGURATIONS)

In [ ]:
if RUN_BUILD:
    for index, cfg in enumerate(CONFIGURATIONS):
        build_stage1_from_manifest(
            manifest=cfg['manifest'],
            output=STAGE1,
            dataset_name=cfg['dataset'],
            subsampled_cell_count=cfg['cells'],
            seed=SEED,
            mode='w' if index == 0 else 'a',
        )
        print(f"built {cfg['dataset']}/{cfg['cells']}")
else:
    print('No HDF5 written. Set RUN_BUILD=True after all previews and audits pass.')

## 7. Verify the Stage 1 file and summarize every split

This checks the stored schema and reports patient/tube coverage plus stored and original event counts. A stored count may be below the requested level when the source file contains fewer events.

In [ ]:
if STAGE1.exists():
    stage1_report = audit_stage1(STAGE1)
    print(stage1_report.summary())
    statistics = []
    with h5py.File(STAGE1, 'r') as handle:
        for dataset_name in handle:
            for cell_level in handle[dataset_name]:
                for sample_key, sample in handle[dataset_name][cell_level].items():
                    for tube_id, tube in sample.items():
                        matrix = tube['raw_cell_matrix']
                        statistics.append({
                            'dataset': dataset_name, 'cell_level': cell_level,
                            'patient_id': tube.attrs.get('patient_id', sample_key),
                            'label': tube.attrs.get('label'), 'tube_id': tube_id,
                            'stored_cells': matrix.shape[0], 'markers': matrix.shape[1],
                            'original_count': int(tube.attrs['counts']),
                        })
    stats = pd.DataFrame(statistics)
    display(stats.head())
    display(stats.groupby(['dataset', 'cell_level', 'tube_id']).agg(
        patients=('patient_id', 'nunique'),
        labels=('label', 'nunique'),
        stored_min=('stored_cells', 'min'),
        stored_median=('stored_cells', 'median'),
        original_min=('original_count', 'min'),
        original_max=('original_count', 'max'),
    ).reset_index())
else:
    print(f"No Stage 1 file yet: {STAGE1.resolve()}")

## Troubleshooting

- **FCS import error**: install the `fcs` extra and restart the kernel.
- **Unmatched files**: inspect relative paths in the discovery cell and adjust `filename_pattern`; do not silently relabel them.
- **Missing labels**: ensure IDs in the label CSV exactly match IDs captured from filenames (including leading zeros).
- **Marker-count mismatch**: the declared marker list must have one entry per matrix column.
- **Mixed panels**: map marker names separately under each `tube_id`.
- **Need to rebuild**: delete/move the output deliberately or use a new output name; safeguards prevent accidental replacement.